# Exercises for lesson 3 
---

## Data Loaders:

In [49]:
import pandas as pd;
import plotly.express as px;
import plotly.graph_objects as go;
from plotly.subplots import make_subplots;

heartFrame = pd.read_csv('Data/Heart.csv');
collegeFrame = pd.read_csv('Data/College.csv');
wageFrame = pd.read_csv('Data/Wage.csv');
autoFrame = pd.read_csv('Data/Auto.csv');
bikeFrame = pd.read_csv('Data/Bikeshare.csv');

## Exercise 1: Heart Disease Dataset

### Heart Disease Data Dictionary
| Feature | Data Type | Description |
| :--- | :--- | :--- |
| **age** | Integer | Patient's age in years. |
| **sex** | Integer | Biological sex (1 = male; 0 = female). |
| **cp** | Integer | Chest pain type (0=Typical angina, 1=Atypical, 2=Non-anginal, 3=Asymptomatic). |
| **trestbps** | Integer | Resting blood pressure (mm Hg) upon hospital admission. |
| **chol** | Integer | Serum cholesterol level (mg/dl). |
| **fbs** | Integer | Fasting blood sugar > 120 mg/dl (1 = true; 0 = false). |
| **restecg** | Integer | Resting electrocardiogram results (0=Normal, 1=ST-T abnormality, 2=Hypertrophy). |
| **thalach** | Integer | Maximum heart rate achieved during exercise stress test. |
| **exang** | Integer | Exercise-induced angina (1 = yes; 0 = no). |
| **oldpeak** | Float | ST depression induced by exercise relative to rest. |
| **slope** | Integer | Slope of the peak exercise ST segment (0=Upsloping, 1=Flat, 2=Downsloping). |
| **ca** | Integer | Number of major vessels (0-3) colored by fluoroscopy. |
| **thal** | Integer | Thalassemia blood flow proxy (1=Normal, 2=Fixed defect, 3=Reversible defect). |
| **target** | Integer | Presence of heart disease (1 = Disease present; 0 = No disease). |

### Questions:
1) Does stress test heartrate drop with age, and how does that correlate with a heart disease diagnosis?
2) Is typical angina most likely to indicate heart disease, or does atypical/asymtomatic have a higher diagnosis rate?
3) Are high colesteral and high blood preassure really the sign of heart disease or can we more accurately perdict with stress tests

### Story Telling:
 a) Are we looking at the right signs to precidct heart disease, or are we missing the real signs?\
 b) Cholesterol and age often get the blame for heart disease, is this really the case?\
 c) We often find that being asymptomatic has a much higher rate than expected of heart disease\
 d) frame around a specific imaginary person going for a check-up, what should they be checked for?\
 e) Start by showing how much less helpful colesteral is than people think, reveal how much more important stress tests are\. 

In [50]:
# Plots for Heart Disease
source_text = "Source: Kaggle Heart Disease Dataset";

# Create a readable label column for the legends
heartFrame['Diagnosis'] = heartFrame['target'].map({0: 'No Disease', 1: 'Disease Present'});

# Chart 1: Heart Rate vs Age
# Simple insight into the general groups who should worry
fig1 = px.scatter(heartFrame, x='age', y='thalach', color='Diagnosis', opacity=0.7, color_discrete_map={'No Disease': '#1f77b4', 'Disease Present': '#d62728'});
fig1.update_layout(title='Max Heart Rate (Stress Test) by Age and Diagnosis', xaxis_title='Age (Years)', yaxis_title='Maximum Heart Rate Achieved (thalach)');
fig1.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig1.show();

# Chart 2: Chest Pain and Diagnosis
# this one shows us the less obvious truth of what types of pain symptoms we should really worry about 
# Map numeric cp to string labels for the chart
# Updated map for 1-4 indexing
cp_map = {1: 'Typical Angina', 2: 'Atypical Angina', 3: 'Non-anginal', 4: 'Asymptomatic'};
heartFrame['CP_Type'] = heartFrame['cp'].map(cp_map);

# Group and calculate percentages for 100% stacked bar
cp_props = heartFrame.groupby(['CP_Type', 'Diagnosis']).size().reset_index(name='count');
cp_props['percentage'] = cp_props.groupby('CP_Type')['count'].transform(lambda x: x / x.sum() * 100);

fig2 = px.bar(cp_props, x='CP_Type', y='percentage', color='Diagnosis', color_discrete_map={'No Disease': '#1f77b4', 'Disease Present': '#d62728'});
fig2.update_layout(title='Heart Disease Diagnosis Rate by Chest Pain Type', xaxis_title='Chest Pain Type', yaxis_title='Percentage (%)', barmode='stack');
fig2.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig2.show();

# Chart 3: Cholesterol & Blood Pressure Boxplots
# this teaches us how much less chol and BP matter than weve been led to believe they do 
fig3 = make_subplots(rows=1, cols=2, subplot_titles=('Cholesterol Levels', 'Resting Blood Pressure'));

# Cholesterol Box Traces
fig3.add_trace(go.Box(y=heartFrame[heartFrame['target'] == 0]['chol'], name='No Disease', marker_color='#1f77b4'), row=1, col=1);
fig3.add_trace(go.Box(y=heartFrame[heartFrame['target'] == 1]['chol'], name='Disease Present', marker_color='#d62728'), row=1, col=1);

# Blood Pressure Box Traces
fig3.add_trace(go.Box(y=heartFrame[heartFrame['target'] == 0]['trestbps'], name='No Disease', marker_color='#1f77b4', showlegend=False), row=1, col=2);
fig3.add_trace(go.Box(y=heartFrame[heartFrame['target'] == 1]['trestbps'], name='Disease Present', marker_color='#d62728', showlegend=False), row=1, col=2);

fig3.update_layout(title_text='Evaluating Traditional Risk Factors: Cholesterol and Blood Pressure', boxmode='group');
fig3.update_yaxes(title_text="Cholesterol (mg/dl)", row=1, col=1);
fig3.update_yaxes(title_text="Resting BP (mm Hg)", row=1, col=2);
fig3.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig3.show();

# Chart 4: Correlation Barplot
# this serves as an overview as to what risk factors matter and how 
correlations = heartFrame[['age', 'chol', 'trestbps', 'thalach', 'oldpeak', 'target']].corr()['target'].drop('target').sort_values();
corr_df = correlations.reset_index();
corr_df.columns = ['Metric', 'Correlation'];

metric_names = {
    'oldpeak': 'ST Depression after Exercise (oldpeak)',
    'age': 'Age',
    'trestbps': 'Resting Blood Pressure',
    'chol': 'Cholesterol',
    'thalach': 'Max Heart Rate Achieved (thalach)'
};
corr_df['Metric'] = corr_df['Metric'].map(metric_names);

# Set colors based on positive/negative correlation
corr_df['Color'] = corr_df['Correlation'].apply(lambda x: '#d62728' if x < 0 else '#2ca02c');

fig4 = px.bar(corr_df, x='Correlation', y='Metric', orientation='h', color='Color', color_discrete_map='identity');
fig4.update_layout(title='Correlation of Clinical Metrics with Heart Disease Diagnosis', xaxis_title='Correlation Coefficient (Pearson)', yaxis_title='Clinical Metric');
fig4.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));

fig4.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(size=10, color='#2ca02c'), legendgroup='Pos', showlegend=True, name='Positive (Higher = More Risk)'));
fig4.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(size=10, color='#d62728'), legendgroup='Neg', showlegend=True, name='Negative (Lower = More Risk)'));

fig4.show();

### Insights: 
This data shows us a couple interesting things, first off it shows us that the things we normally associate with being risk factors for heart disease arent actually very good predictors, IE cholesterol and blood pressure, however things like thalach and oldpeak are much better predictors. And most interesting is that typical angina is actually less likely to result in a heart disease diagnosis compared to non-anginal or atypical pain, but it also requires you to look past the overwhelming amount of heart disease in the aymptomatic catogory, due to those people being at the clinic already for other reasons or risk factors we dont have here in this set, it forces you to look deeper into where the data comes from. 

---
# Exercise 2: College Dataset
### Data Dictionary (ISLR College Dataset)
| Feature | Data Type | Description |
| :--- | :--- | :--- |
| **Private** | Categorical | Public or private university indicator (Yes/No). |
| **Apps** | Integer | Number of applications received. |
| **Accept** | Integer | Number of applications accepted. |
| **Enroll** | Integer | Number of new students enrolled. |
| **Top10perc** | Integer | Pct. of new students from top 10% of high school class. |
| **Top25perc** | Integer | Pct. of new students from top 25% of high school class. |
| **F.Undergrad** | Integer | Number of full-time undergraduates. |
| **P.Undergrad** | Integer | Number of part-time undergraduates. |
| **Outstate** | Integer | Out-of-state tuition (dollars). |
| **Room.Board** | Integer | Room and board costs (dollars). |
| **Books** | Integer | Estimated book costs (dollars). |
| **Personal** | Integer | Estimated personal spending (dollars). |
| **PhD** | Integer | Pct. of faculty with Ph.D.'s. |
| **Terminal** | Integer | Pct. of faculty with terminal degrees. |
| **S.F.Ratio** | Float | Student/faculty ratio. |
| **perc.alumni** | Integer | Pct. of alumni who donate. |
| **Expend** | Integer | Instructional expenditure per student (dollars). |
| **Grad.Rate** | Integer | Graduation rate. |

### Questions:
1) Does paying higher out-of-state tuition actually correlate with a higher graduation rate, or do cheaper schools perform just as well?
2) How does the student-to-faculty ratio differ between private and public universities, and does it impact graduation rates?
3) Is instructional expenditure the best predictor of graduation success, or do other factors matter more?

### Storytelling Plan:
* **a)** Start with a question: "Do you really get what you pay for in higher education?"
* **b)** Challenge assumptions: We often assume expensive private colleges guarantee better outcomes, but let's see if the data backs that up.
* **c)** Use surprise as your weapon: Reveal that spending more per student hits diminishing returns, and expose a famous quirk in this specific dataset (a school with a graduation rate over 100%).
* **d)** Make data human: Frame the analysis around a local high school senior from Seattle trying to decide if taking on debt for a private out-of-state school is really worth it.
* **e)** Build a narrative arc: Setup the cost vs. public/private debate -> Reveal the relationship between tuition and graduation -> Discover what factors *actually* correlate with graduation success -> Conclude on what matters for our prospective student.

In [51]:
source_text = "Source: ISLR College Dataset (1995)";

# fix weird error 
collegeFrame.loc[collegeFrame['Grad.Rate'] > 100, 'Grad.Rate'] = 100;
collegeFrame.rename(columns={'Unnamed: 0': 'College'}, inplace=True);

# Chart 1: Tuition vs Graduation Rate
# This gives you a good overview of graduation rate with type of college, to frame the rest of the data with 
fig1 = px.scatter(collegeFrame, x='Outstate', y='Grad.Rate', color='Private', opacity=0.7, color_discrete_map={'Yes': '#4B2E83', 'No': '#b7a57a'}, hover_data=['College']);
fig1.update_layout(title='Do You Get What You Pay For? Tuition vs. Graduation Rate', xaxis_title='Out-of-State Tuition ($)', yaxis_title='Graduation Rate (%)');
fig1.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig1.show();

# Chart 2: Student-to-Faculty Ratios by Institution Type
# this shows how distinct the class sizes are, which is a well known big difference in student success 
fig2 = px.box(collegeFrame, x='Private', y='S.F.Ratio', color='Private', color_discrete_map={'Yes': '#4B2E83', 'No': '#b7a57a'}, points="all", hover_data=['College']);
fig2.update_layout(title='Class Sizes: Private vs. Public Universities', xaxis_title='Private Institution', yaxis_title='Student to Faculty Ratio');
fig2.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig2.show();

# Chart 3: Expenditure vs Success 
# this mainly serves to show if the success of a school is more about the school, or who it admits. 
fig3 = px.scatter(collegeFrame, x='Expend', y='Grad.Rate', color='Private', size='Top10perc', color_discrete_map={'Yes': '#4B2E83', 'No': '#b7a57a'}, hover_data=['College', 'Top10perc']);
fig3.update_layout(title='Instructional Expenditure per Student vs Graduation Rate', xaxis_title='Instructional Expenditure ($)', yaxis_title='Graduation Rate (%)');
fig3.add_annotation(text="Bubble size = % of students from Top 10% of HS class", xref="paper", yref="paper", x=0, y=1.05, showarrow=False, font=dict(size=11, color="black"));
fig3.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig3.show();

# Chart 4: What Actually Predicts Graduation?
# this is our big final all of the stats chart, showing us what matters the most 
# Grab only numeric columns for correlation
numeric_cols = collegeFrame.select_dtypes(include=['number']).columns;
correlations = collegeFrame[numeric_cols].corr()['Grad.Rate'].drop('Grad.Rate').sort_values();
corr_df = correlations.reset_index();
corr_df.columns = ['Metric', 'Correlation'];

metric_names = {
    'Outstate': 'Out-of-State Tuition',
    'Top10perc': 'Top 10% of HS Class',
    'Top25perc': 'Top 25% of HS Class',
    'Expend': 'Instructional Expenditure',
    'Room.Board': 'Room & Board Costs',
    'perc.alumni': 'Alumni Donation Rate',
    'Terminal': 'Faculty w/ Terminal Degree',
    'PhD': 'Faculty w/ PhD',
    'S.F.Ratio': 'Student/Faculty Ratio',
    'Accept': 'Total Accepted',
    'Apps': 'Total Applications',
    'F.Undergrad': 'Full-Time Undergrads',
    'P.Undergrad': 'Part-Time Undergrads'
};
corr_df['Metric'] = corr_df['Metric'].map(metric_names).fillna(corr_df['Metric']);

# Set colors based on positive/negative
corr_df['Color'] = corr_df['Correlation'].apply(lambda x: '#d62728' if x < 0 else '#2ca02c');

fig4 = px.bar(corr_df, x='Correlation', y='Metric', orientation='h', color='Color', color_discrete_map='identity');
fig4.update_layout(title='What Drives Graduation? Correlation with Graduation Rate', xaxis_title='Correlation Coefficient (Pearson)', yaxis_title='University Metric');
fig4.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig4.show();

### Insights:
These charts show that while the things people often think of as being important like class size, education of staff, and cost are important, one of the largest factors in the success of a college is the incoming students, in the final chart we can see that in the top 4 most relevant stats, are both top 10% of hs class, and top 25%, showing how strog of an initiator it really is. 

---

## Exercise 3: Wage Dataset
### Data Dictionary (ISLR Wage Dataset)
| Feature | Data Type | Description |
| :--- | :--- | :--- |
| **year** | Integer | Year that wage information was recorded (2003-2009). |
| **age** | Integer | Age of worker. |
| **maritl** | Categorical | Marital status (1. Never Married, 2. Married, 3. Widowed, 4. Divorced, 5. Separated). |
| **race** | Categorical | Race (1. White, 2. Black, 3. Asian, 4. Other). |
| **education** | Categorical | Education level (1. < HS Grad, 2. HS Grad, 3. Some College, 4. College Grad, 5. Advanced Degree). |
| **region** | Categorical | Region of the country (mostly Mid-Atlantic in this subset). |
| **jobclass** | Categorical | Type of job (1. Industrial, 2. Information). |
| **health** | Categorical | Health status of worker (1. <=Good, 2. >=Very Good). |
| **health_ins** | Categorical | Does the worker have employer-provided health insurance? (1. Yes, 2. No). |
| **logwage** | Float | Logarithm of the workers' wage. |
| **wage** | Float | Workers' raw wage in thousands of dollars. |

### Questions:
1) What is the exact monetary value of the "education premium," and does a college degree raise your earning floor or just your earning ceiling?
2) Do wages continue to climb steadily with age, or is there an eventual plateau depending on whether you work in the Information or Industrial sector?
3) Is employer-provided health insurance standard across the board, or is there a "double penalty" where lower-wage job classes also lack benefits?

### Storytelling Plan:
* **a) Start with a question:** "Is staying in school for an advanced degree actually worth the investment, or does industry matter more?"
* **b) Challenge assumptions:** We assume you just make more money as you get older, but we'll test if age actually protects you from hitting an income ceiling.
* **c) Use surprise as a weapon:** Reveal the compounding disparity between high earners getting health insurance while lower earners are left paying out of pocket.
* **d) Make data human:** Frame the story around a local college senior deciding whether to jump straight into the tech (Information) industry or stay for a Master's degree.
* **e) Build a narrative arc:** Establish the value of education -> Look at how that plays out over a lifespan (Age) -> Split the data by industry (Information vs Industrial) -> Deliver the final blow about total compensation (Wage + Insurance).
* 

In [52]:
source_text = "Source: ISLR Wage Dataset (Mid-Atlantic Male Workers)";

edu_order = ['1. < HS Grad', '2. HS Grad', '3. Some College', '4. College Grad', '5. Advanced Degree'];

# Chart 1: The Education Premium 
# this give us the baseline we need to interpret this data, of how much college education matters (answer : a lot) 
fig1 = px.box(wageFrame, x='education', y='wage', color='education', category_orders={'education': edu_order});
fig1.update_layout(title='The Education Premium: Wage Distribution by Degree', xaxis_title='Education Level', yaxis_title='Wage (Thousands of $)');
fig1.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig1.show();

# Chart 2: Age vs Wage by Sector 
# this shows us that there is a large upwards trend for people in the information sector as they age, more so than industrial 
fig2 = px.scatter(wageFrame, x='age', y='wage', color='jobclass', opacity=0.6, color_discrete_map={'1. Industrial': '#b7a57a', '2. Information': '#4B2E83'});
fig2.update_layout(title='Career Trajectory: Income Ceiling by Age and Sector', xaxis_title='Age of Worker', yaxis_title='Wage (Thousands of $)');
fig2.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig2.show();

# Chart 3: Health Insurance by Education 
# this gives us a new perspective on the original chart, but with how many people get insurance 
ins_props = wageFrame.groupby(['education', 'health_ins']).size().reset_index(name='count');
ins_props['percentage'] = ins_props.groupby('education')['count'].transform(lambda x: x / x.sum() * 100);

fig3 = px.bar(ins_props, x='education', y='percentage', color='health_ins', barmode='stack', category_orders={'education': edu_order}, color_discrete_map={'1. Yes': '#2ca02c', '2. No': '#d62728'});
fig3.update_layout(title='Benefits Access: Health Insurance Rates by Education', xaxis_title='Education Level', yaxis_title='Percentage (%)');
fig3.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig3.show();

# Chart 4: The Double Penalty 
# this shows us that information workers are heavily clustered at higher wages and generally receive health insurance, while Industrial workers without insurance are heavily clustered at the lowest wage brackets.
fig4 = px.violin(wageFrame, x='health_ins', y='wage', color='jobclass', box=True, color_discrete_map={'1. Industrial': '#b7a57a', '2. Information': '#4B2E83'});
fig4.update_layout(title='The Double Penalty: Wage Distribution by Insurance Access', xaxis_title='Has Employer Health Insurance?', yaxis_title='Wage (Thousands of $)', violinmode='group');
fig4.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig4.show();

## Insights:
This data if nothing else shows is very clearly that education is the biggest factor in how much money you will make, more so than anything else, while also showing us that in general people tend to be better off in the information sector, particularly as they age. 

---

## Exercise 4: Auto Dataset
### Data Dictionary (ISLR Auto Dataset)

| Feature | Data Type | Description |
| :--- | :--- | :--- |
| **mpg** | Float | Miles per gallon (fuel efficiency). |
| **cylinders** | Integer | Number of engine cylinders (between 3 and 8). |
| **displacement** | Float | Engine displacement (in cubic inches). |
| **horsepower** | Float | Engine horsepower. |
| **weight** | Float | Vehicle weight (in pounds). |
| **acceleration** | Float | Time to accelerate from 0 to 60 mph (in seconds). |
| **year** | Integer | Model year (modulo 100, so 70 means 1970). |
| **origin** | Categorical | Origin of car (1 = American, 2 = European, 3 = Japanese). |
| **name** | String | Vehicle name/make and model. |

### Questions:
1) What is the ultimate enemy of fuel efficiency: the raw weight of the car, or the size (displacement/horsepower) of the engine?
2) How did the different global markets (American, European, Japanese) stack up against each other during the height of the 1970s oil crisis?
3) Did the automotive industry actually innovate over this 12-year period, or were they just building smaller, weaker cars to save gas?

### Storytelling Plan:
* **a) Start with a question:** "When gas prices skyrocket, do we have to sacrifice power for efficiency, or can engineering save us?"
* **b) Challenge assumptions:** We often assume 1970s American muscle cars were the only gas guzzlers, but let's look at the global landscape.
* **c) Use surprise as a weapon:** Reveal that acceleration actually *improved* slightly over the decade, even as cars got wildly more fuel-efficient.
* **d) Make data human:** Put the audience in the shoes of a 1975 car buyer panicking at the pump—should they buy local or import?
* **e) Build a narrative arc:** Establish the basic physics of MPG -> Show the regional battle for efficiency -> Reveal the timeline of how the industry engineered its way out of the crisis.



In [53]:
source_text = "Source: ISLR Auto Dataset (1970-1982)";

autoFrame['horsepower'] = pd.to_numeric(autoFrame['horsepower'], errors='coerce');
autoFrame = autoFrame.dropna(subset=['horsepower']);

origin_map = {1: 'American', 2: 'European', 3: 'Japanese'};
autoFrame['Origin'] = autoFrame['origin'].map(origin_map);

region_colors = {'American': '#4B2E83', 'European': '#b7a57a', 'Japanese': '#1f77b4'};

# Chart 1: The Physics of Efficiency 
# establishes our baseline of where cars lie, and which cars are in which part 
fig1 = px.scatter(autoFrame, x='weight', y='mpg', color='Origin', size='horsepower', hover_data=['name'], color_discrete_map=region_colors, opacity=0.7);
fig1.update_layout(title='The Enemy of Efficiency: Vehicle Weight vs. MPG', xaxis_title='Vehicle Weight (lbs)', yaxis_title='Miles Per Gallon (MPG)');
fig1.add_annotation(text="Bubble size = Engine Horsepower", xref="paper", yref="paper", x=0, y=1.05, showarrow=False, font=dict(size=11, color="black"));
fig1.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig1.show();

# Chart 2: The Regional Battle 
# this shows us the very clear dominance from the japanese in MPG, and how far behind america is 
fig2 = px.box(autoFrame, x='Origin', y='mpg', color='Origin', points="all", color_discrete_map=region_colors, hover_data=['name']);
fig2.update_layout(title='Fuel Economy by Region During the Oil Crisis', xaxis_title='Manufacturing Region', yaxis_title='Miles Per Gallon (MPG)');
fig2.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig2.show();

# Chart 3: Innovation Over Time 
#this one shows us some interesting facts, mainly that Europe did overtake japan at the end of the dataset, but also that all countries had a dip in 72-73  
yearly_mpg = autoFrame.groupby(['year', 'Origin'])['mpg'].mean().reset_index();
yearly_mpg['Year_Full'] = yearly_mpg['year'] + 1900;

fig3 = px.line(yearly_mpg, x='Year_Full', y='mpg', color='Origin', markers=True, color_discrete_map=region_colors);
fig3.update_layout(title='Industry Evolution: Average MPG over the 1970s', xaxis_title='Model Year', yaxis_title='Average MPG');
fig3.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig3.show();

# Chart 4: What Predicts Fuel Efficiency? 
# finale that shows what the actual culprits behind ad MPG are 
numeric_cols = autoFrame.select_dtypes(include=['number']).columns;
correlations = autoFrame[numeric_cols].corr()['mpg'].drop(['mpg', 'origin']).sort_values();
corr_df = correlations.reset_index();
corr_df.columns = ['Metric', 'Correlation'];

corr_df['Color'] = corr_df['Correlation'].apply(lambda x: '#d62728' if x < 0 else '#2ca02c');

fig4 = px.bar(corr_df, x='Correlation', y='Metric', orientation='h', color='Color', color_discrete_map='identity');
fig4.update_layout(title='What Destroys Fuel Economy? (Correlation with MPG)', xaxis_title='Correlation Coefficient (Pearson)', yaxis_title='Vehicle Metric');
fig4.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig4.show();

## Insights:
these show us some interesting facts, firstly it shows us sheer dominance overall by the japanese in MPG, as well as showing us how severely lacking the US was. we can also see that while the US was always behind, the europe was gaining on japan, traded a few blows, and did pass them in the final year of the data. But it also shows us that while MPG did go up year by year, the main things holding it back were still weight and engine size, meaning the US either cared less, or couldnt fix those as effectively. 

---

## Exercise 5: Bikeshare Dataset
### Data Dictionary (UCI Bikeshare Dataset - Hourly)

| Feature | Data Type | Description |
| :--- | :--- | :--- |
| **instant** | Integer | Record index. |
| **dteday** | String/Date | Date of the record. |
| **season** | Categorical | Season (1 = Spring, 2 = Summer, 3 = Fall, 4 = Winter). |
| **yr** | Categorical | Year (0 = 2011, 1 = 2012). |
| **mnth** | Integer | Month (1 to 12). |
| **hr** | Integer | Hour of the day (0 to 23). |
| **holiday** | Categorical | Whether the day is a holiday (1 = Yes, 0 = No). |
| **weekday** | Integer | Day of the week (0 to 6). |
| **workingday** | Categorical | If day is neither weekend nor holiday (1 = Yes, 0 = No). |
| **weathersit** | Categorical | Weather situation (1=Clear/Cloudy, 2=Mist, 3=Light Rain/Snow, 4=Heavy Rain/Ice). |
| **temp** | Float | Normalized temperature in Celsius. |
| **atemp** | Float | Normalized "feels like" temperature. |
| **hum** | Float | Normalized humidity. |
| **windspeed** | Float | Normalized wind speed. |
| **casual** | Integer | Count of casual (unregistered) users. |
| **registered** | Integer | Count of registered (subscribed) users. |
| **cnt** | Integer | Total count of total rental bikes (casual + registered). |

### Questions:
1) How does the time of day dictate ridership, and does this pattern completely change between working days and weekends?
2) Do environmental factors (temperature and weather conditions) affect casual tourists and registered commuters the same way?
3) If a city needs to pull a large portion of the bike fleet offline for maintenance, when is the exact best time to do so?

### Storytelling Plan:
* **a) Start with a question:** "Are bikeshare systems primarily tourist attractions, or are they vital arteries for daily commuters?"
* **b) Challenge assumptions:** We might assume people only ride bikes when the weather is perfect, but let's see how weather actually impacts the dedicated commuter.
* **c) Use surprise as a weapon:** Reveal the massive, undeniable "double peak" in the hourly data that perfectly outlines the 9-to-5 corporate workday.
* **d) Make data human:** Frame the data from the perspective of a city planner trying to figure out when to schedule fleet maintenance without causing a commuter meltdown.
* **e) Build a narrative arc:** Establish the daily heartbeat of the city (Hourly) -> Break down the demographics (Registered vs Casual) -> Introduce the elements (Weather) -> Give the city planner their actionable conclusion.
*

In [54]:
source_text = "Source: UCI Bike Sharing Dataset (Hourly)";

bikeFrame.rename(columns={'bikers': 'cnt'}, inplace=True);
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'};
bikeFrame['Season'] = bikeFrame['season'].map(season_map);
day_map = {0: 'Weekend/Holiday', 1: 'Working Day'};
bikeFrame['Day_Type'] = bikeFrame['workingday'].map(day_map);
bikeFrame['Weather'] = bikeFrame['weathersit'];

#  Chart 1: The Commuter Pulse 
# this shows us that while weekdays have clear work commute spike (with a small lunch spike) weekends are smooth and peak mid day
hourly_avg = bikeFrame.groupby(['hr', 'Day_Type'])['cnt'].mean().reset_index();
fig1 = px.line(hourly_avg, x='hr', y='cnt', color='Day_Type', markers=True, color_discrete_map={'Working Day': '#4B2E83', 'Weekend/Holiday': '#b7a57a'});
fig1.update_layout(title='The Commuter Pulse: Average Hourly Ridership', xaxis_title='Hour of the Day (0-23)', yaxis_title='Average Bike Rentals');
fig1.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig1.update_xaxes(tickmode='linear', dtick=1);
fig1.show();

#  Chart 2: Who rides when? 
# this shows us how the seasons effect ridership, while also adding in the extra info of who is riding 
season_users = bikeFrame.groupby('Season')[['casual', 'registered']].sum().reset_index();
season_users['Season'] = pd.Categorical(season_users['Season'], categories=['Spring', 'Summer', 'Fall', 'Winter'], ordered=True);
season_users = season_users.sort_values('Season');
fig2 = px.bar(season_users, x='Season', y=['casual', 'registered'], barmode='stack', color_discrete_map={'casual': '#b7a57a', 'registered': '#4B2E83'});
fig2.update_layout(title='User Demographics: Casual vs. Registered Riders by Season', xaxis_title='Season', yaxis_title='Total Rentals', legend_title='User Type');
fig2.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig2.show();

# Chart 3: The Weather Effect 
# this shows us how impactful the different weather is  
fig3 = px.box(bikeFrame, x='Weather', y='cnt', color='Weather');
fig3.update_layout(title='Fair Weather Riders? Ridership Distribution by Weather Condition', xaxis_title='Weather Situation', yaxis_title='Hourly Bike Rentals');
fig3.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig3.show();

# Chart 4: Correlation Matrix 
# this gives us our final tell all of what actually effects ridership the most 
numeric_cols = bikeFrame.select_dtypes(include=['number']).columns;
correlations = bikeFrame[numeric_cols].corr()['cnt'].drop(['cnt', 'casual', 'registered', 'Unnamed: 0']).sort_values();
corr_df = correlations.reset_index();
corr_df.columns = ['Metric', 'Correlation'];
metric_names = {'hr': 'Hour of Day', 'temp': 'Temperature', 'atemp': 'Feels Like Temp', 'yr': 'Year', 'mnth': 'Month', 'season': 'Season', 'windspeed': 'Wind Speed', 'workingday': 'Working Day', 'weekday': 'Day of Week', 'holiday': 'Holiday', 'hum': 'Humidity', 'weathersit': 'Bad Weather Level'};
corr_df['Metric'] = corr_df['Metric'].map(metric_names).fillna(corr_df['Metric']);
corr_df['Color'] = corr_df['Correlation'].apply(lambda x: '#d62728' if x < 0 else '#2ca02c');
fig4 = px.bar(corr_df, x='Correlation', y='Metric', orientation='h', color='Color', color_discrete_map='identity');
fig4.update_layout(title='What Drives Rentals? Correlation with Hourly Count', xaxis_title='Correlation Coefficient (Pearson)', yaxis_title='System Variable');
fig4.add_annotation(text=source_text, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10, color="gray"));
fig4.show();

## Insights:
this shows us while some of the stats are more obvious, like snow meaning less ridership, or time of day, there are some unexpected findings, particularly that spring ridership isnt much more than 1/3rd of fall ridership, despite rainy and cloudy days being more common, however if you dig further into the data youll find that this is because of how seasons were named in the data, being so far off, that you could basically move each one back a place and it would be more correct, meaning falls glory, is in reality summers. 